# 05 · 100 epoch validation 결과 모아보기

학습된 모델의 validation 기준 best 체크포인트 요약만 읽습니다. test split은 열지 않습니다. 모델별 학습률이 저자 저장소 설정을 따라 다르므로, 표는 동일 학습 recipe 경쟁 결과가 아니라 초기 경향 확인 자료입니다.

In [ ]:
from pathlib import Path
import json, sys, torch
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/CSI_SLLM_RESEARCH')
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT_OVERRIDE = None  # 필요하면 프로젝트 경로 지정
if PROJECT_ROOT_OVERRIDE: PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE)
runs = {
    'CsiNet': PROJECT_ROOT / 'runs' / 'csinet_indoor_cr4' / 'best.pt',
    'CRNet': PROJECT_ROOT / 'runs' / 'crnet_indoor_cr4_100ep' / 'summary.json',
    'TransNet': PROJECT_ROOT / 'runs' / 'transnet_indoor_cr4_100ep' / 'summary.json',
}
rows = []
for name, path in runs.items():
    if not path.is_file():
        rows.append((name, 'not run yet', '-', '-'))
        continue
    if path.suffix == '.json':
        result = json.loads(path.read_text(encoding='utf-8'))
        rows.append((name, str(result['best_epoch']), f"{result['best_val_nmse_db']:.3f}", f"{result['elapsed_seconds']/60:.1f} min"))
    else:
        result = torch.load(path, map_location='cpu', weights_only=True)
        db = result.get('val_nmse_db')
        if db is None: db = 10 * torch.log10(torch.tensor(result['val_nmse_linear'])).item()
        rows.append((name, str(result.get('epoch', '?')), f'{db:.3f}', 'not recorded'))
print(f"{'Model':<12} {'Best epoch':<12} {'Val NMSE dB':<14} {'Train time'}")
print('-' * 56)
for row in rows: print(f'{row[0]:<12} {row[1]:<12} {row[2]:<14} {row[3]}')

비교표의 숫자는 validation 결과입니다. 모델 선택·학습 recipe 조정에만 사용하고, 최종 논문 수치나 test 결과로 보고하지 마세요. CsiNet의 100 epoch 실행은 이전 노트북에서 저장한 체크포인트를 읽습니다.